# Liquidity Capacity Analysis
## How Much Until the Edge Starts Decaying?

**Fee Hurdle:** 0.10% (10 bps) for market-order round-trip (0.05% per leg)

**Symbols:** BTCPERP (perp) / BTCUSDT (spot)

This notebook compares:
- **Phase 1 (Static):** Walk-the-book VWAP at signal time, no book modification
- **Phase 2 (Dynamic):** Live LOB replay with actual liquidity consumption

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

# Fee hurdle
FEE_HURDLE_BPS = 0.0010  # 10 bps
FEE_HURDLE_BPS_STR = '10 bps'

DATA_DIR = Path('research/data')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
# Load results if they exist
static_path = DATA_DIR / 'liquidity_capacity_results.parquet'
dynamic_path = DATA_DIR / 'liquidity_capacity_dynamic.parquet'

df_static = None
df_dynamic = None

if static_path.exists():
    df_static = pd.read_parquet(static_path)
    print(f"Loaded static results: {len(df_static):,} rows")
else:
    print("Static results not found. Run liquidity_capacity.py first.")

if dynamic_path.exists():
    df_dynamic = pd.read_parquet(dynamic_path)
    print(f"Loaded dynamic results: {len(df_dynamic):,} rows")
else:
    print("Dynamic results not found. Run liquidity_capacity_dynamic.py first.")

---
## 1. Edge Decay Curve

The fundamental question: **as order size increases, how does the net captured spread decay?**

The fee hurdle (10 bps) is shown as a horizontal line. The point where the curve crosses this line is the **maximum executable size**.

In [ ]:
def plot_edge_decay(df_static, df_dynamic=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- Panel 1: Net Spread in USD ---
    ax = axes[0]
    
    if df_static is not None:
        static_means = df_static.groupby('qty')['net_spread'].agg(['mean', 'median'])
        ax.plot(static_means.index, static_means['mean'], 'o-', label='Static (mean)', color='#2196F3', linewidth=2)
        ax.plot(static_means.index, static_means['median'], 's--', label='Static (median)', color='#1976D2', linewidth=1.5)
    
    if df_dynamic is not None:
        dynamic_means = df_dynamic.groupby('qty')['net_spread'].agg(['mean', 'median'])
        ax.plot(dynamic_means.index, dynamic_means['mean'], 'o-', label='Dynamic (mean)', color='#F44336', linewidth=2)
        ax.plot(dynamic_means.index, dynamic_means['median'], 's--', label='Dynamic (median)', color='#D32F2F', linewidth=1.5)
    
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('Net Captured Spread (USD)', fontsize=12)
    ax.set_title('Edge Decay: Net Spread vs Order Size', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # --- Panel 2: Net Spread in bps ---
    ax = axes[1]
    
    if df_static is not None:
        ref_price = df_static.groupby('qty')['perp_vwap'].mean()
        static_bps = (static_means['mean'] / ref_price) * 10000
        static_bps_med = (static_means['median'] / ref_price) * 10000
        ax.plot(static_bps.index, static_bps, 'o-', label='Static (mean)', color='#2196F3', linewidth=2)
        ax.plot(static_bps_med.index, static_bps_med, 's--', label='Static (median)', color='#1976D2', linewidth=1.5)
    
    if df_dynamic is not None:
        ref_price_dyn = df_dynamic.groupby('qty')['perp_vwap'].mean()
        dynamic_bps = (dynamic_means['mean'] / ref_price_dyn) * 10000
        dynamic_bps_med = (dynamic_means['median'] / ref_price_dyn) * 10000
        ax.plot(dynamic_bps.index, dynamic_bps, 'o-', label='Dynamic (mean)', color='#F44336', linewidth=2)
        ax.plot(dynamic_bps_med.index, dynamic_bps_med, 's--', label='Dynamic (median)', color='#D32F2F', linewidth=1.5)
    
    # Fee hurdle line
    ax.axhline(y=FEE_HURDLE_BPS * 10000, color='green', linestyle='-', linewidth=2, 
               label=f'Fee Hurdle ({FEE_HURDLE_BPS_STR})')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('Net Captured Spread (bps)', fontsize=12)
    ax.set_title('Edge Decay: Net Spread in bps vs Order Size', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'edge_decay_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_edge_decay(df_static, df_dynamic)

---
## 2. Depth Exhaustion Probability

What fraction of signals have insufficient depth to fill a given order size?

In [ ]:
def plot_depth_exhaustion(df_static, df_dynamic=None):
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if df_static is not None:
        static_exh = df_static.groupby('qty')['depth_exhausted'].mean() * 100
        ax.plot(static_exh.index, static_exh, 'o-', label='Static', color='#2196F3', linewidth=2, markersize=8)
    
    if df_dynamic is not None:
        dynamic_exh = df_dynamic.groupby('qty')['depth_exhausted'].mean() * 100
        ax.plot(dynamic_exh.index, dynamic_exh, 's-', label='Dynamic', color='#F44336', linewidth=2, markersize=8)
    
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('P(Depth Exhausted) (%)', fontsize=12)
    ax.set_title('Probability of Depth Exhaustion vs Order Size', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 105)
    
    # Add a 50% threshold line
    ax.axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='50% threshold')
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'depth_exhaustion.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_depth_exhaustion(df_static, df_dynamic)

---
## 3. Per-Signal Capacity Distribution

For each signal, what is the maximum order size that still yields a positive net spread?

In [ ]:
def compute_signal_capacity(df):
    """For each signal, find max qty where net_spread > 0."""
    capacities = []
    for sig_time in df['signal_time'].unique():
        sub = df[df['signal_time'] == sig_time].dropna(subset=['net_spread'])
        if len(sub) == 0:
            continue
        positive = sub[sub['net_spread'] > 0]
        if len(positive) > 0:
            capacities.append(positive['qty'].max())
        else:
            capacities.append(0)  # No positive size
    return pd.Series(capacities)

def plot_capacity_distribution(df_static, df_dynamic=None):
    fig, axes = plt.subplots(1, 2 if df_dynamic is not None else 1, figsize=(14, 6))
    if df_dynamic is None:
        axes = [axes]
    
    for ax, (df, label, color) in zip(axes, [
        (df_static, 'Static', '#2196F3'),
        (df_dynamic, 'Dynamic', '#F44336')
    ]):
        if df is None:
            continue
        caps = compute_signal_capacity(df)
        
        # Histogram
        ax.hist(caps, bins=20, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
        ax.axvline(caps.median(), color='darkred', linestyle='--', linewidth=2, 
                   label=f'Median: {caps.median():.2f} BTC')
        ax.axvline(caps.mean(), color='darkblue', linestyle=':', linewidth=2,
                   label=f'Mean: {caps.mean():.2f} BTC')
        
        ax.set_xlabel('Max Profitable Size (BTC)', fontsize=12)
        ax.set_ylabel('Number of Signals', fontsize=12)
        ax.set_title(f'{label} - Per-Signal Capacity Distribution', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Stats box
        stats_text = (
            f"P25: {caps.quantile(0.25):.2f} BTC\n"
            f"P50: {caps.median():.2f} BTC\n"
            f"P75: {caps.quantile(0.75):.2f} BTC\n"
            f"Zero capacity: {(caps == 0).mean():.1%}"
        )
        ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'capacity_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_capacity_distribution(df_static, df_dynamic)

---
## 4. Static vs Dynamic Comparison

The **slippage penalty** is the difference between static and dynamic results. 
A negative penalty means the dynamic simulation (with live book consumption) 
produced worse fills than the static walk — this is the cost of our own market impact.

In [ ]:
def plot_slippage_penalty(df_static, df_dynamic):
    if df_static is None or df_dynamic is None:
        print("Both static and dynamic results needed for comparison.")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    static_means = df_static.groupby('qty')['net_spread'].mean()
    dynamic_means = df_dynamic.groupby('qty')['net_spread'].mean()
    
    # --- Panel 1: Absolute penalty ---
    ax = axes[0]
    penalty = dynamic_means - static_means
    colors = ['#4CAF50' if p >= 0 else '#F44336' for p in penalty]
    ax.bar(penalty.index.astype(str), penalty.values, color=colors, alpha=0.8, edgecolor='white')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('Slippage Penalty (USD)', fontsize=12)
    ax.set_title('Slippage Penalty: Dynamic - Static (USD)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # --- Panel 2: Penalty in bps ---
    ax = axes[1]
    ref_price = df_static.groupby('qty')['perp_vwap'].mean()
    penalty_bps = (penalty / ref_price) * 10000
    colors = ['#4CAF50' if p >= 0 else '#F44336' for p in penalty_bps]
    ax.bar(penalty_bps.index.astype(str), penalty_bps.values, color=colors, alpha=0.8, edgecolor='white')
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('Slippage Penalty (bps)', fontsize=12)
    ax.set_title('Slippage Penalty: Dynamic - Static (bps)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'slippage_penalty.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_slippage_penalty(df_static, df_dynamic)

---
## 5. Capacity Heatmap (Time x Size)

Shows how capacity varies over time. Are there periods where liquidity is systematically lower?

In [ ]:
def plot_capacity_heatmap(df, title='Capacity Heatmap'):
    if df is None:
        return
    
    # Pivot: for each signal, what's the net spread at each size?
    # We'll aggregate by hour of day
    df_plot = df.copy()
    df_plot['hour'] = pd.to_datetime(df_plot['signal_time']).dt.hour
    
    # Compute mean net spread by hour and qty
    pivot = df_plot.pivot_table(
        values='net_spread',
        index='hour',
        columns='qty',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create heatmap
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
    
    # Labels
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{h:02d}:00' for h in pivot.index])
    
    ax.set_xlabel('Order Size (BTC)', fontsize=12)
    ax.set_ylabel('Hour of Day', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean Net Spread (USD)', fontsize=11)
    
    # Overlay fee hurdle contour
    # Find where net spread crosses zero
    
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'capacity_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_capacity_heatmap(df_static, 'Static: Net Spread by Hour and Size')
if df_dynamic is not None:
    plot_capacity_heatmap(df_dynamic, 'Dynamic: Net Spread by Hour and Size')

---
## 6. Summary Table

Key metrics for each order size.

In [ ]:
def generate_summary_table(df_static, df_dynamic=None):
    """Generate a comprehensive summary table."""
    
    if df_static is None:
        return
    
    rows = []
    for qty in sorted(df_static['qty'].unique()):
        row = {'Qty (BTC)': qty}
        
        for label, df in [('Static', df_static), ('Dynamic', df_dynamic)]:
            if df is None:
                continue
            sub = df[df['qty'] == qty].dropna(subset=['net_spread'])
            if len(sub) == 0:
                row[f'{label} N'] = 0
                row[f'{label} Mean Spread'] = np.nan
                row[f'{label} Median Spread'] = np.nan
                row[f'{label} P(Positive)'] = np.nan
                row[f'{label} P(Exhausted)'] = np.nan
            else:
                ref_price = sub['perp_vwap'].mean()
                row[f'{label} N'] = len(sub)
                row[f'{label} Mean Spread'] = sub['net_spread'].mean()
                row[f'{label} Median Spread'] = sub['net_spread'].median()
                row[f'{label} Mean bps'] = (sub['net_spread'].mean() / ref_price) * 10000
                row[f'{label} P(Positive)'] = (sub['net_spread'] > 0).mean()
                row[f'{label} P(Exhausted)'] = sub['depth_exhausted'].mean()
        
        rows.append(row)
    
    df_summary = pd.DataFrame(rows)
    
    # Format for display
    display_cols = ['Qty (BTC)']
    for label in ['Static', 'Dynamic']:
        if df_dynamic is None and label == 'Dynamic':
            continue
        display_cols += [
            f'{label} N',
            f'{label} Mean Spread',
            f'{label} Median Spread',
            f'{label} Mean bps',
            f'{label} P(Positive)',
            f'{label} P(Exhausted)',
        ]
    
    display_df = df_summary[display_cols].copy()
    
    # Format numbers
    for col in display_df.columns:
        if 'Spread' in col and 'bps' not in col:
            display_df[col] = display_df[col].map('${:,.2f}'.format)
        elif 'bps' in col:
            display_df[col] = display_df[col].map('{:.2f}'.format)
        elif 'P(' in col:
            display_df[col] = display_df[col].map('{:.1%}'.format)
    
    return display_df

summary = generate_summary_table(df_static, df_dynamic)
if summary is not None:
    display(summary)
    summary.to_csv(DATA_DIR / 'liquidity_capacity_summary.csv', index=False)
    print("\nSummary saved to research/data/liquidity_capacity_summary.csv")

---
## 7. Conclusions & Recommendations

Based on the analysis above, we can answer:

1. **What is the maximum safe order size?**  
   The largest size where P(net spread > 0) >= 50% and median net spread > fee hurdle.

2. **How much does market impact cost us?**  
   The difference between static and dynamic results quantifies the cost of our own footprint.

3. **When is liquidity best/worst?**  
   The heatmap shows intraday patterns in available capacity.

4. **What's the optimal position sizing rule?**  
   Size = min(signal_strength * base_size, max_capacity_for_this_time)